### Reload models ###

In [4]:
import pandas as pd 
import numpy as np 
import joblib 
import matplotlib.pyplot as plt 
import seaborn as sns 
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve)

log_reg = joblib.load('../models/logistic_regression_model.joblib')
tfidf = joblib.load('../models/tfidf_vectorizer.joblib')
log_reg_mc = joblib.load('../models/scam_type_model.joblib')
tfidf_mc = joblib.load('../models/tfidf_vectorizer_multiclass.joblib')

In [7]:
from scipy.sparse import hstack, csr_matrix

count_cols = ['message_length', 'word_count', 'exclamation_count', 'caps_word_count']
flag_cols = ['has_urgency', 'has_authority_claim', 'has_financial_request',
             'has_credential_request', 'has_url', 'has_phone_number', 'has_email']

def build_manual(d):
    X = d[count_cols + flag_cols].copy()
    X[count_cols] = np.log1p(X[count_cols])
    return csr_matrix(X.values)


test_df = pd.read_csv('../data/processed/test_binary.csv')
test_df['processed_message'] = test_df['processed_message'].fillna('')

X_test = hstack([tfidf.transform(test_df['processed_message']), build_manual(test_df)]).tocsr()
y_test = (test_df['label'] == 'scam').astype(int)

test_mc_df = pd.read_csv('../data/processed/test_multiclass.csv')
X_test_mc_tfidf = tfidf_mc.transform(test_mc_df['processed_message'].fillna(''))
y_test_mc = test_mc_df['scam_type']

#### Generate predictions for both models ####

In [8]:
y_prob_lr = log_reg.predict_proba(X_test)[:, 1]
FINAL_THRESHOLD = 0.4
y_pred_lr = (y_prob_lr >= FINAL_THRESHOLD).astype(int)

y_pred_mc = log_reg_mc.predict(X_test_mc_tfidf)

In [9]:
print(classification_report(y_test, y_pred_lr, target_names=['legit', 'scam']))
print(classification_report(y_test_mc, y_pred_mc))

              precision    recall  f1-score   support

       legit       0.98      0.95      0.97      7105
        scam       0.87      0.95      0.91      2369

    accuracy                           0.95      9474
   macro avg       0.93      0.95      0.94      9474
weighted avg       0.95      0.95      0.95      9474

                precision    recall  f1-score   support

   banking_kyc       0.84      0.98      0.90       197
digital_arrest       0.92      0.86      0.89        14
    investment       0.13      0.31      0.19        13
      job_scam       0.94      0.95      0.95       144
       lottery       0.67      0.82      0.74        57
      phishing       0.99      0.91      0.95      1283
   upi_payment       0.19      0.77      0.30        13

      accuracy                           0.92      1721
     macro avg       0.67      0.80      0.70      1721
  weighted avg       0.95      0.92      0.93      1721

